# 09 — General pricing schema and GPT-5 extraction

**Goal:** send different providers' PDF text through the same extraction workflow, with no provider-name parser required. Each result has the same structure, exact source quotes, and a clear next step.

This lesson builds the general **extraction and review stage**. It does not yet replace Notebook 08's calculator or make every extracted rule executable. A later lesson will connect reviewed rules to calculation engines and your 12 months of usage.

Start with `MODE = "example"`. The two offline examples are **manually authored fictional page text and answers**, not real PDFs or GPT-5 results. They let you learn the schema without an API key. Then choose `live` for your PDFs or `replay` for pricing records saved by this notebook.

Use your project's `.venv` kernel and run sections in order. You do not need to rerun Notebooks 01–08 first.

## 1. Imports and paths
The reusable implementation lives in `pricing_schema.py`, `pricing_extraction.py`, and `pricing_examples.py`. Check that the interpreter below belongs to `.venv`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import json
import sys
from IPython.display import HTML, display
from dotenv import load_dotenv

from electricity_optimizer.ingestion import read_pdf
from electricity_optimizer.pricing_schema import (
    PricingCandidate, PlanIdentity, assess_pricing, normalized_kwh_rate,
)
from electricity_optimizer.pricing_extraction import (
    PricingRecord, build_pricing_workflow, load_pricing_record, save_pricing_record,
)
from electricity_optimizer.pricing_examples import teaching_examples

ROOT = Path.cwd()
CONTRACTS = ROOT / "contracts"
OUTPUT = ROOT / "output/lesson09"
RECORDS = OUTPUT / "records"
if not (ROOT / "electricity_optimizer").is_dir():
    raise FileNotFoundError("Open this notebook from the project root.")

def show_table(rows, columns):
    header = "".join(f"<th>{escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Python:", sys.executable)
print("Available PDFs:")
for path in sorted(CONTRACTS.glob("*")):
    if path.is_file() and path.suffix.lower() == ".pdf":
        print(" -", path.name)

Python: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\.venv\Scripts\python.exe
Available PDFs:
 - 2106-00134_Terms_and_Conditions_of_Supply_-_1_June_2021.pdf
 - EA_MKTTC_DL_0224_FINAL_Digital.pdf
 - GetContractVersionPDF_143273.pdf
 - R1F00160000021A.pdf
 - R1F00166003766B.pdf
 - R1F00169972621A.pdf
 - TOSA.pdf


## 2. Understand the shared schema

A **schema** specifies the fields every provider's extracted result must contain. Pydantic checks types and structure. Our additional checks verify quotes and some consistency rules. Neither proves that every clause has been interpreted correctly.

| Part | What it stores |
|---|---|
| `identity` | Provider, plan, country, territory, currency, customer type, date and contract terms |
| `coverage` | Whether each charge category is present, explicitly absent, missing or ambiguous |
| `rules` | Original amounts/units, rule types, thresholds, tiers, time windows, conditions and citations |
| `unresolved_terms` | Anything that cannot be represented confidently |
| `supporting_documents_needed` | Missing rate schedules or other supporting material |

`not_stated` means **unknown**, never zero. `explicit_none` needs evidence explicitly ruling out a charge. Monetary amounts are decimal strings to preserve precision. We retain `10 cents_per_kwh` and let Python convert it to `0.10 currency_per_kwh`.

Tiered, time-of-use, demand and indexed rules can be recorded for review, but their full calculators are future work. Complex conditions remain text for review; the program does not execute model-generated expressions.

In [2]:
schema = PricingCandidate.model_json_schema()
show_table([{"field": name, "description": details.get("description", "Structured field")}
            for name, details in schema["properties"].items()], ["field", "description"])
print("Required identity fields:", ", ".join(PlanIdentity.model_fields))

field,description
identity,Structured field
coverage,"One entry for every component, including missing components."
rules,Structured field
unresolved_terms,Structured field
supporting_documents_needed,Structured field


Required identity fields: provider, plan_name, country, territory, currency, customer_type, document_date, contract_length, renewal_terms, termination_terms


## 3. Choose example, live or replay mode

Leave **example** selected for your first run.

- `example`: two authored fictional examples; no key and no network calls.
- `live`: sends extracted text/table cells from the selected PDFs to **GPT-5**. This incurs API usage. Rerunning Section 4 in live mode makes new calls. Begin with one PDF.
- `replay`: loads **Notebook 09 pricing records**, checks their PDF fingerprints and reruns validation with no API calls. Old Notebook 02 agreement JSONs use a different schema and cannot be replayed here.

For live mode, put your key in the existing `.env`, change `MODE` below, and rerun Sections 3–7. The key is loaded only in live mode and is never printed. For replay, paste the saved record paths printed by Section 4 into `REPLAY_FILES`. Replay never falls back to live calls.

All top-level PDFs can use this same extractor. To select them all, replace `SELECTED_PDFS` with the commented expression, after trying one file.

In [13]:
MODE = "live"  # "example", "live", or "replay"
MODEL = "gpt-5"
SELECTED_PDFS = ["R1F00160000021A.pdf"]
# SELECTED_PDFS = sorted(p.name for p in CONTRACTS.iterdir() if p.is_file() and p.suffix.lower() == ".pdf")
REPLAY_FILES = []  # Example: ["output/lesson09/records/pricing_<id>.json"]

if MODE not in {"example", "live", "replay"}:
    raise ValueError("Choose example, live, or replay.")
if MODE == "live":
    load_dotenv(ROOT / ".env", override=False)
    print("LIVE: Section 4 will send", len(SELECTED_PDFS), "selected PDF(s) to", MODEL)
elif MODE == "replay":
    print("REPLAY: only saved Notebook 09 records will be read.")
else:
    print("EXAMPLE: manually authored fictional data; no API calls.")

LIVE: Section 4 will send 1 selected PDF(s) to gpt-5


## 4. Run extraction → validation → supervisor

The LangGraph workflow has one LLM extraction step in live mode. Validation and the supervisor's routing are deterministic Python functions. These are foundations for the final multi-agent system, not a claim that the entire system is finished.

The supervisor chooses among four next steps:

| Route | Meaning |
|---|---|
| `needs_evidence_review` | A quote, page, field or rule-consistency check failed |
| `needs_information` | Required context or prices are unknown/ambiguous |
| `needs_pricing_engine` | Rules require additional calculation logic or usage detail |
| `candidate_for_pricing_review` | Initial checks passed; interpretation still needs review before calculation |

Each successful live result is saved immediately to its own record file. Failures are reported per PDF, so one bad document does not hide the other results. Input is limited to 150,000 characters including table cells; oversized PDFs are rejected without truncation. OCR, chunking and automatic multi-document linking are later extensions.

In [14]:
runs, failures = [], []
workflow = build_pricing_workflow(mode="live" if MODE == "live" else "replay", model=MODEL)

def selected_pdf(name):
    path = (CONTRACTS / name).resolve()
    if path.parent != CONTRACTS.resolve() or path.suffix.lower() != ".pdf" or not path.is_file():
        raise ValueError("Select an existing top-level PDF in contracts/.")
    return path

if MODE == "example":
    for document, record in teaching_examples():
        state = workflow.invoke({"document": document, "record": record})
        runs.append({"state": state, "saved_record": None})
else:
    selections = SELECTED_PDFS if MODE == "live" else REPLAY_FILES
    if not selections:
        raise ValueError("Select PDFs for live mode or Notebook 09 record paths for replay, then rerun Section 3.")
    for selection in dict.fromkeys(selections):
        try:
            if MODE == "live":
                document = read_pdf(selected_pdf(selection), include_tables=True)
                state = workflow.invoke({"document": document})
                saved = save_pricing_record(state["record"], RECORDS)
            else:
                saved = (ROOT / selection).resolve()
                header = PricingRecord.model_validate_json(saved.read_text(encoding="utf-8"))
                if header.origin != "live":
                    raise ValueError("Use example mode for authored examples; replay expects a saved live PDF extraction.")
                document = read_pdf(selected_pdf(header.source_file), include_tables=True)
                record = load_pricing_record(saved, document)
                state = workflow.invoke({"document": document, "record": record})
            runs.append({"state": state, "saved_record": str(saved)})
            print("Saved/replayed record:", saved)
        except Exception as exc:
            failures.append({"selection": selection, "error": str(exc)})

show_table([{"source": r["state"]["document"].source_file,
             "origin": r["state"]["record"].origin,
             "route": r["state"]["route"], "approved": False} for r in runs],
           ["source", "origin", "route", "approved"])
if failures:
    show_table(failures, ["selection", "error"])
print("Completed:", len(runs), "Failed:", len(failures))

Saved/replayed record: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson09\records\pricing_2c33d002d0d349efbf4002305d0615a4.json


source,origin,route,approved
R1F00160000021A.pdf,live,needs_information,False


Completed: 1 Failed: 0


## 5. Inspect one result and its evidence

Change `RESULT_INDEX` to select another result. For a real PDF, open the named source and compare the quoted page with the amount, unit, boundary and conditions. For example mode, the authored source text is printed below.

The first example should reach `candidate_for_pricing_review`. The second should need a pricing engine and usage by time period. They use the same schema despite different pricing structures.

The per-kWh conversion column is only a unit demonstration, not a calculated bill. A matching quote does not prove that an extracted amount is correct or that omitted charges do not exist.

In [15]:
RESULT_INDEX = 0
if runs:
    if not 0 <= RESULT_INDEX < len(runs):
        raise ValueError("RESULT_INDEX must identify one of the completed results.")
    selected = runs[RESULT_INDEX]["state"]
    candidate = selected["record"].candidate
    print("Source:", selected["document"].source_file)
    print("Route:", selected["route"])
    if MODE == "example":
        print("\nAUTHORED SOURCE TEXT:\n", selected["document"].pages[0].text)
    show_table([{"field": name, "status": getattr(candidate.identity, name).status,
                 "value": getattr(candidate.identity, name).value,
                 "evidence": [c.model_dump() for c in getattr(candidate.identity, name).citations]}
                for name in PlanIdentity.model_fields], ["field", "status", "value", "evidence"])
    show_table([c.model_dump() for c in candidate.coverage], ["component", "status", "explanation", "citations"])
    rows = []
    for rule in candidate.rules:
        converted = str(normalized_kwh_rate(rule)) if rule.amount is not None and rule.unit in {"cents_per_kwh", "currency_per_kwh"} else "n/a"
        rows.append({"rule": rule.label, "kind": rule.kind, "amount": rule.amount,
                     "unit": rule.unit, "currency/kWh": converted,
                     "conditions": rule.conditions, "time window": rule.time_window,
                     "lower kWh": rule.lower_kwh, "lower inclusive": rule.lower_inclusive,
                     "upper kWh": rule.upper_kwh, "upper inclusive": rule.upper_inclusive,
                     "bands": [b.model_dump() for b in rule.bands],
                     "evidence": [c.model_dump() for c in rule.citations]})
    show_table(rows, ["rule", "kind", "amount", "unit", "currency/kWh", "lower kWh", "lower inclusive", "upper kWh", "upper inclusive", "bands", "time window", "conditions", "evidence"])
    assessment = selected["assessment"]
    show_table(assessment["issues"], ["severity", "field", "message"])
    for reason in assessment["missing_information"] + assessment["engine_work"]:
        print(" -", reason)
    print(assessment["next_step"])
else:
    print("No successful results. Review Section 4 failures before continuing.")

Source: R1F00160000021A.pdf
Route: needs_information


field,status,value,evidence
provider,found,"Reliant Energy Retail Services, LLC","[{'page_number': 1, 'quote': 'Reliant Energy Retail Services, LLC'}]"
plan_name,found,"Reliant Power Savings 2,000 kWh 24 plan","[{'page_number': 1, 'quote': 'Reliant Power Savings 2,000 kWh 24 plan'}]"
country,not_stated,None,[]
territory,found,AEP Texas Central service area,"[{'page_number': 1, 'quote': 'AEP Texas Central service area'}]"
currency,not_stated,None,[]
customer_type,not_stated,None,[]
document_date,found,Date: 09/01/2026,"[{'page_number': 1, 'quote': 'Date: 09/01/2026'}]"
contract_length,found,24 months,"[{'page_number': 1, 'quote': '24 months'}]"
renewal_terms,not_stated,None,[]
termination_terms,found,"Yes. $295. Applies through the end of the contract term. This fee does not apply if the customer moves, and provides a forwarding address and other evidence that may be requested to verify that the customer moved.","[{'page_number': 1, 'quote': 'Yes. $295. Applies through the end of the contract term.'}, {'page_number': 1, 'quote': 'This fee does not apply if the customer moves, and provides a forwarding address and other evidence that may be requested to verify that the customer moved.'}]"


component,status,explanation,citations
energy,present,Energy Charge stated per kWh.,"[{'page_number': 1, 'quote': 'Energy Charge: 15.4124¢ per kWh'}]"
base,present,Base Charge stated per billing cycle.,"[{'page_number': 1, 'quote': 'Base Charge: $0.00 per billing cycle'}]"
delivery,present,TDU delivery charges stated; includes pass-through terms and ITR rider note; special McAllen/Mission exception noted.,"[{'page_number': 1, 'quote': 'AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh'}, {'page_number': 1, 'quote': 'TDU Delivery Charges include all recurring charges from TDU passed through without mark-up and may include an Income Tax Refund Rider (ITR). The ITR refund amount varies based on usage.'}, {'page_number': 1, 'quote': '**Customers in the McAllen/Mission area formerly served by Oncor will not be assessed TC-3, NDC, or SRC charges or the ADFIT credit associated with the SRC.'}]"
credit,present,"Usage Credit of $150.00 per billing cycle when usage ≥ 2,000 kWh; no credit below 2,000 kWh.","[{'page_number': 1, 'quote': 'A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh.'}, {'page_number': 1, 'quote': 'There is no Usage Credit for a billing cycle when usage on this plan is below 2,000 kWh.'}, {'page_number': 1, 'quote': 'Usage Credit: $150.00'}]"
minimum_charge,not_stated,No minimum bill or minimum charge is mentioned on the supplied pages.,[]
demand,not_stated,No demand charge is mentioned on the supplied pages.,[]
tax,not_stated,No taxes or tax surcharges are itemized on the supplied pages.,[]
termination,present,Early termination fee stated with exception for moving.,"[{'page_number': 1, 'quote': 'Yes. $295. Applies through the end of the contract term.'}, {'page_number': 1, 'quote': 'This fee does not apply if the customer moves, and provides a forwarding address and other evidence that may be requested to verify that the customer moved.'}]"
other,present,Other possible fees referenced to Terms of Service but not listed here.,"[{'page_number': 1, 'quote': 'What other fees may I be charged?'}, {'page_number': 1, 'quote': 'For other fees, please reference these paragraphs in the Terms of Service.'}]"


rule,kind,amount,unit,currency/kWh,lower kWh,lower inclusive,upper kWh,upper inclusive,bands,time window,conditions,evidence
Energy Charge,flat_per_kwh,15.4124,cents_per_kwh,0.154124,None,None,None,None,[],None,None,"[{'page_number': 1, 'quote': 'Energy Charge: 15.4124¢ per kWh'}]"
Base Charge,fixed,0.00,currency_per_billing_cycle,n/a,None,None,None,None,[],None,None,"[{'page_number': 1, 'quote': 'Base Charge: $0.00 per billing cycle'}]"
TDU Delivery Charges - monthly,fixed,3.24,currency_per_billing_cycle,n/a,None,None,None,None,[],None,AEP Texas Central. TDU charges are passed through; may include ITR. McAllen/Mission exception applies.,"[{'page_number': 1, 'quote': 'AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh'}, {'page_number': 1, 'quote': 'TDU Delivery Charges include all recurring charges from TDU passed through without mark-up and may include an Income Tax Refund Rider (ITR). The ITR refund amount varies based on usage.'}, {'page_number': 1, 'quote': '**Customers in the McAllen/Mission area formerly served by Oncor will not be assessed TC-3, NDC, or SRC charges or the ADFIT credit associated with the SRC.'}]"
TDU Delivery Charges - usage,flat_per_kwh,5.7554,cents_per_kwh,0.057554,None,None,None,None,[],None,AEP Texas Central. TDU charges are passed through; may include ITR. McAllen/Mission exception applies.,"[{'page_number': 1, 'quote': 'AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh'}, {'page_number': 1, 'quote': 'TDU Delivery Charges include all recurring charges from TDU passed through without mark-up and may include an Income Tax Refund Rider (ITR). The ITR refund amount varies based on usage.'}, {'page_number': 1, 'quote': '**Customers in the McAllen/Mission area formerly served by Oncor will not be assessed TC-3, NDC, or SRC charges or the ADFIT credit associated with the SRC.'}]"
ITR refund (varies),other,None,unknown,n/a,None,None,None,None,[],None,Income Tax Refund Rider amount varies based on usage and is passed through from the TDU.,"[{'page_number': 1, 'quote': 'may include an Income Tax Refund Rider (ITR). The ITR refund amount varies based on usage.'}]"
"Usage Credit ≥ 2,000 kWh",usage_credit,150.00,currency_per_billing_cycle,n/a,2000,True,None,None,[],None,"Included per billing cycle when usage on this plan is above or equal to 2,000 kWh; no Usage Credit when usage is below 2,000 kWh.","[{'page_number': 1, 'quote': 'A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh.'}, {'page_number': 1, 'quote': 'There is no Usage Credit for a billing cycle when usage on this plan is below 2,000 kWh.'}, {'page_number': 1, 'quote': 'Usage Credit: $150.00'}]"
Early Termination Fee,fixed,295,currency,n/a,None,None,None,None,[],None,Applies through the end of the contract term. Waived if the customer moves and provides a forwarding address and requested evidence.,"[{'page_number': 1, 'quote': 'Yes. $295. Applies through the end of the contract term.'}, {'page_number': 1, 'quote': 'This fee does not apply if the customer moves, and provides a forwarding address and other evidence that may be requested to verify that the customer moved.'}]"
Other fees (see Terms of Service),other,None,unknown,n/a,None,None,None,None,[],None,"Refer to Terms of Service (Pricing, Billing, Payment and Payment Arrangement, Customer Care, Alternate Billing and Payment Options).","[{'page_number': 1, 'quote': 'What other fees may I be charged?'}, {'page_number': 1, 'quote': 'For other fees, please reference these paragraphs in the Terms of Service.'}, {'page_number': 1, 'quote': 'Pricing, Billing, Payment and Payment Arrangement, Customer Care, Alternate Billing and Payment Options'}]"


severity,field,message


 - Resolve identity.country before grouping plans.
 - Resolve identity.currency before grouping plans.
 - Resolve identity.customer_type before grouping plans.
 - Resolve minimum_charge: No minimum bill or minimum charge is mentioned on the supplied pages.
 - Resolve demand: No demand charge is mentioned on the supplied pages.
 - Resolve rule: ITR refund (varies).
 - Resolve rule: Other fees (see Terms of Service).
 - TDU pass-through may include an Income Tax Refund Rider (ITR) whose refund amount varies based on usage; the document does not provide a numeric value.
 - Customers in the McAllen/Mission area formerly served by Oncor have exceptions to certain TDU charges (TC-3, NDC, SRC) and the ADFIT credit associated with the SRC; exact impacts are not priced here.
 - Price may change during the contract period only to reflect changes in TDSP charges, ERCOT/TRE administrative fees, or new/modified fees or costs from laws or regulatory actions; magnitude of such changes is not specifie

## 6. Try a deliberately incorrect quote (offline)

This exercise changes a **copy of the fictional example**, then checks it again. It should route to `needs_evidence_review` because the cited rate never appeared in the source. Your live results are unaffected.

This test catches a fabricated quote. It cannot catch every semantic error—for example, citing the correct sentence but assigning its number to the wrong charge category. That is why review/evaluation remains a separate step.

In [11]:
example_document, example_record = teaching_examples()[0]
changed = example_record.candidate.model_copy(deep=True)
energy_rule = next(r for r in changed.rules if r.component == "energy")
energy_rule.citations[0].quote = "Energy: 99 cents per kWh at all hours."
exercise = assess_pricing(changed, example_document)
print("Deliberately incorrect quote ->", exercise["route"])
show_table(exercise["issues"], ["severity", "field", "message"])

Deliberately incorrect quote -> needs_evidence_review


severity,field,message
error,rules[1] All-hours energy,Quote not found on page 1.


## 7. Save the review summary

The summary includes successful candidates, source fingerprints, issues, next steps and failures. It records whether the input was an authored example, live extraction or replay. Saving does not approve any plan. Live record files from Section 4 can be replayed even if you never run this section.

In [16]:
OUTPUT.mkdir(parents=True, exist_ok=True)
report = {
    "lesson": "09_general_pricing_extraction", "mode": MODE,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "approved_for_comparison": False,
    "scope": "Extraction and review routing only; no cost calculation or ranking.",
    "results": [{"record": r["state"]["record"].model_dump(mode="json"),
                 "assessment": r["state"]["assessment"], "saved_record": r["saved_record"]}
                for r in runs],
    "failures": failures,
}
summary_path = OUTPUT / f"{MODE}_pricing_review_summary.json"
summary_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print("Next: inspect the evidence, then connect reviewed standard rules to the cost calculator.")

Saved: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson09\live_pricing_review_summary.json
Next: inspect the evidence, then connect reviewed standard rules to the cost calculator.


## What this adds to the final project

The upload workflow can now ask GPT-5 for the same pricing structure across providers. Provider-specific regex adapters are no longer required for this extraction stage. Missing documents and unsupported calculations still need explicit handling.

Next, we will turn a supported subset of reviewed rules into executable pricing, compare compatible plans against the same 12 months of usage, and feed those results to recommendation/negotiation agents. Time-of-use and demand plans will require finer usage data. The full multi-agent supervisor and upload interface remain later work.

Implementation references: [OpenAI Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs) and [GPT-5](https://developers.openai.com/api/docs/models/gpt-5). The SDK parses the response into Pydantic models; source and pricing validation are our separate responsibilities.